# Part 2: Cross-Domain Acne Classification

**Self-contained Colab notebook** — no need to run Part 1 first.

Train a binary **acne vs clear-skin** classifier on **ACNE04 patches** (ResNet-50), then evaluate on **DermNet** with test-time domain adaptation and Grad-CAM.

**Before you start**
- Runtime → **GPU** (T4 is enough; ~30–60 min end-to-end)
- Free [Roboflow API key](https://app.roboflow.com/settings/api) (ACNE04 download)
- [Kaggle API token](https://www.kaggle.com/settings) (`kaggle.json`) for DermNet

**Test-time adaptation ablation** (same trained weights; varies preprocessing only):
1. Baseline (`da=none`)
2. + Histogram matching
3. + Reinhard color normalization
4. + Reinhard + TTA (horizontal flip)

## 0. Setup

Link: https://github.com/nprakash1/acne_yang_project/tree/main/part2_classification

In [ ]:
# Auto-detect how the project got into this Colab runtime.
# Supports three workflows -- you don't have to edit anything:
#   (1) git clone   - set GITHUB_URL below to a public repo URL
#   (2) zip upload  - drag acne_yang_project.zip into Colab's file pane
#   (3) Drive copy  - put the unzipped folder anywhere in your Drive
import os, glob, subprocess

GITHUB_URL = 'https://github.com/nprakash1/acne_yang_project.git'
PROJECT_DIR = 'acne_yang_project'

def _is_project_root(p):
    return os.path.isdir(os.path.join(p, 'part2_classification')) and \
           os.path.isfile(os.path.join(p, 'requirements.txt'))

# Already at project root?
if _is_project_root('.'):
    print('[setup] already at project root')
elif os.path.isdir(PROJECT_DIR) and _is_project_root(PROJECT_DIR):
    %cd $PROJECT_DIR
    print(f'[setup] cd into ./{PROJECT_DIR}')
else:
    zips = glob.glob('/content/*.zip') + glob.glob('./*.zip')
    drive_hits = glob.glob('/content/drive/MyDrive/**/part2_classification', recursive=True)
    if zips:
        z = zips[0]
        print(f'[setup] unzipping {z}')
        subprocess.check_call(['unzip', '-q', '-o', z, '-d', '/content/_extracted'])
        roots = [r for r, _, _ in os.walk('/content/_extracted') if _is_project_root(r)]
        assert roots, 'zip did not contain a project root (part2_classification/ + requirements.txt)'
        %cd $roots[0]
    elif drive_hits:
        root = os.path.dirname(drive_hits[0])
        print(f'[setup] using project from Drive: {root}')
        %cd $root
    elif GITHUB_URL:
        print(f'[setup] cloning {GITHUB_URL}')
        subprocess.check_call(['git', 'clone', GITHUB_URL])
        %cd $PROJECT_DIR
    else:
        raise RuntimeError(
            'No project source found. Either:\n'
            ' - set GITHUB_URL above to your repo, or\n'
            ' - drag acne_yang_project.zip into the Colab file pane and re-run, or\n'
            ' - mount Drive with the project folder inside MyDrive.'
        )

print('[setup] cwd =', os.getcwd())
!pip install -q -r requirements.txt
print('[setup] dependencies installed')

### GPU check (optional Drive)

Enable **Runtime → Change runtime type → GPU** before training.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — training will be very slow. Enable GPU in Runtime settings.')

# Optional: mount Drive to persist outputs across sessions
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

## 1. Download ACNE04 (COCO format)

Patches are built from detection bounding boxes. We only need the **COCO** export (not YOLO).

Get a free API key at https://app.roboflow.com/settings/api — paste when prompted, or set Colab secret `ROBOFLOW_API_KEY`.

In [ ]:
import os
from getpass import getpass
from pathlib import Path

COCO_ANN = Path('data/acne04/coco/train/_annotations.coco.json')

if COCO_ANN.exists():
    print('[data] ACNE04 COCO already present:', COCO_ANN.parent.parent)
else:
    if not os.environ.get('ROBOFLOW_API_KEY'):
        try:
            from google.colab import userdata
            os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
        except Exception:
            pass
    if not os.environ.get('ROBOFLOW_API_KEY'):
        os.environ['ROBOFLOW_API_KEY'] = 'c1orEQIdQdZYPwUiFe2I'
    from part1_detection.data import download_acne04
    coco_root = download_acne04(out_dir='data/acne04', fmt='coco')
    print('[data] COCO root:', coco_root)

assert COCO_ANN.exists(), f'missing {COCO_ANN} — check Roboflow download'

## 2. Generate ACNE04 patches

In [ ]:
from pathlib import Path
from part2_classification.make_patches import make_patches

PATCH_ROOT = Path('data/acne04_patches')
if (PATCH_ROOT / 'train' / 'pos').exists() and any((PATCH_ROOT / 'train' / 'pos').glob('*.jpg')):
    print('[patches] already built at', PATCH_ROOT.resolve(), '— delete folder to rebuild')
else:
    counts = make_patches(
        coco_root='data/acne04/coco',
        out_root=str(PATCH_ROOT),
        expand=1.3, target_size=224, neg_per_pos=1.0,
    )
    print(counts)

## 3. Download DermNet (Kaggle)

Upload `kaggle.json` when prompted (from https://www.kaggle.com/settings → **Create New Token**), or place it in `~/.kaggle/` before running.

In [ ]:
import os
from pathlib import Path

# Replace with your Kaggle credentials
os.environ['KAGGLE_USERNAME'] = "nealprakash"
os.environ['KAGGLE_KEY'] = "KGAT_ed0d3bfd02156b873efc91a96f4f10b6"

DERMNET_ROOT = Path('data/dermnet')
TRAIN_DIR = DERMNET_ROOT / 'train'
TEST_DIR = DERMNET_ROOT / 'test'

if TRAIN_DIR.is_dir() and TEST_DIR.is_dir():
    print('[data] DermNet already present at', DERMNET_ROOT.resolve())
else:
    !kaggle datasets download -d shubhamgoel27/dermnet -p data/dermnet --unzip

for sub in ('train', 'test'):
    p = DERMNET_ROOT / sub
    n_cls = len([d for d in p.iterdir() if d.is_dir()]) if p.is_dir() else 0
    n_img = sum(1 for _ in p.rglob('*') if _.suffix.lower() in {'.jpg', '.jpeg', '.png'}) if p.is_dir() else 0
    print(f'{sub}: {n_cls} class folders, {n_img} images')
    assert p.is_dir(), f'missing {p} — check Kaggle unzip layout'

print('DermNet root:', DERMNET_ROOT.resolve())

## 4. Train classifier on ACNE04

**Model:** ResNet-50 (ImageNet pretrained) + 2-class head. Heavy albumentations during training; best checkpoint by **val AUROC**.

In [ ]:
from part2_classification.train_classifier import train as train_clf
best_weights = train_clf(
    patches_root='data/acne04_patches',
    out_dir='outputs/classifier',
    backbone='resnet50',
    epochs=15,
    heavy_da=True,
)
best_weights

## 5. Domain-gap ablation on DermNet

In [ ]:
from part2_classification.evaluate_dermnet import evaluate as eval_dermnet
import pandas as pd

table = {}
for da, tta, name in [
    ('none', False, 'baseline'),
    ('histogram', False, '+histogram'),
    ('reinhard', False, '+reinhard'),
    ('reinhard', True, '+reinhard+TTA'),
]:
    table[name] = eval_dermnet(
        weights=str(best_weights),
        dermnet_root='data/dermnet',
        da_method=da,
        tta=tta,
        out_dir=f'outputs/dermnet_eval/{name.replace("+","").replace(" ","_")}',
    )
pd.DataFrame(table).T[['acc', 'f1', 'auroc']]

## 6. Grad-CAM on 10 DermNet predictions

In [ ]:
from part2_classification.gradcam import visualize_gradcam
from IPython.display import Image as IPyImage

out = visualize_gradcam(
    weights=str(best_weights),
    predictions_json='outputs/dermnet_eval/reinhardTTA/predictions.json',
    dermnet_train_root='data/dermnet/train',
    da_method='reinhard',
    n_samples=10,
)
IPyImage(filename=str(out))